In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from tqdm import tqdm
import os
os.chdir("/home/pgcp-ai/MachineLearning/Datasets/")

In [10]:
milk = pd.read_csv('milk.csv', index_col = 0)
milk

,water,protein,fat,lactose,ash
Animal,,,,,
HORSE,90.1,2.6,1.0,6.9,0.35
ORANGUTAN,88.5,1.4,3.5,6.0,0.24
MONKEY,88.4,2.2,2.7,6.4,0.18
DONKEY,90.3,1.7,1.4,6.2,0.40
HIPPO,90.4,0.6,4.5,4.4,0.10
CAMEL,87.7,3.5,3.4,4.8,0.71
BISON,86.9,4.8,1.7,5.7,0.90
BUFFALO,82.1,5.9,7.9,4.7,0.78
GUINEA PIG,81.9,7.4,7.2,2.7,0.85


In [48]:
ss = StandardScaler().set_output(transform = 'pandas')
milk_scaled = ss.fit_transform(milk)
milk_scaled.shape

(25, 5)

In [24]:
dbscan = DBSCAN(eps = 0.4, min_samples=2)
dbscan.fit(milk_scaled)
dbscan.labels_

array([-1,  0,  0, -1, -1,  1,  1,  2, -1, -1,  2,  1, -1, -1,  1,  2, -1,
       -1, -1, -1,  3,  3, -1, -1, -1])

In [52]:
epsilon = np.linspace(0.01,2,100)
min_number_of_points = [2,3,4,5]
scores=[]
for n in tqdm(min_number_of_points):
    for e in epsilon:
        db = DBSCAN(eps=e,min_samples=n)
        db.fit(milk_scaled)
        labels = db.labels_
        milk_labeled = milk_scaled.copy()
        milk_labeled['Cluster'] = labels
        milk_labeled_inlier = milk_labeled[milk_labeled['Cluster'] != -1]
        if len(milk_labeled_inlier['Cluster'].unique()) >=2:
            score = silhouette_score(milk_labeled_inlier.drop('Cluster',axis=1),milk_labeled_inlier['Cluster'])
            scores.append([n,e,score])
        else:
            continue
df_scores = pd.DataFrame(scores,columns=['Minimum Number of Points','Epsilon Distance','Silhouette Score'])
df_scores.sort_values('Silhouette Score',ascending=False)


100%|█████████████████████████████████████████████| 4/4 [00:01<00:00,  3.09it/s]


,Minimum Number of Points,Epsilon Distance,Silhouette Score
0,2,0.311515,0.917544
1,2,0.331616,0.903367
2,2,0.351717,0.864756
103,3,0.874343,0.657551
104,3,0.894444,0.657551
...,...,...,...
34,2,0.994949,0.434482
98,3,0.773838,0.418695
96,3,0.733636,0.418695
95,3,0.713535,0.418695


In [53]:
db = DBSCAN(eps=0.994949,min_samples=2)
db.fit(milk_scaled)

DBSCAN(eps=0.994949, min_samples=2)

In [54]:
db.labels_

array([ 0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  0,  0,  1,
       -1, -1,  1,  2,  2,  2, -1, -1])

In [55]:
db = DBSCAN(eps=0.492424,min_samples=2)
db.fit(milk_scaled)

DBSCAN(eps=0.492424, min_samples=2)

In [56]:
db.labels_

array([ 0,  0,  0,  0, -1,  1,  1,  2, -1, -1,  2,  1,  0, -1,  1,  2, -1,
       -1, -1, -1,  3,  3, -1, -1, -1])

In [57]:
from ipywidgets import interact
import ipywidgets as widgets

In [58]:
def dbscan_clust(e, m):
    dbscan = DBSCAN(eps = e, min_samples = m)
    dbscan.fit(milk_scaled)
    return dbscan.labels_

In [59]:
interact(dbscan_clust, e = widgets.FloatSlider(min = 0.01, max = 2, step = 0.01, value = 0.3),
         m = widgets.IntSlider(min = 2, max = 5, step = 1, value = 2))

Widget Javascript not detected.  It may not be installed or enabled properly.


<function __main__.dbscan_clust(e, m)>

In [60]:
interact(dbscan_clust, e = widgets.FloatSlider(min = 0.01, max = 2, step = 0.01, value = 0.3),
         m = [2,3,4,5,6])

Widget Javascript not detected.  It may not be installed or enabled properly.


<function __main__.dbscan_clust(e, m)>